# AI가 대화를 기억하는 방법

## 1. 싱글턴 (Single-turn)
한 번의 질문(입력)과 한 번의 답변(출력)으로 대화 세션이 독립적으로 종료되는 방식입니다.  
AI가 이전의 대화 내용(컨텍스트)이나 맥락을 전혀 기억하지 않습니다.  
특징: 이전 맥락을 고려할 필요가 없어 리소스 소모가 적고 속도가 빠르지만, 연속적인 후속 질문 시 이전에 했던 말을 다시 설명해주어야 합니다.

##### chap03/sec02/single_turn.py

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

while True:
    user_input = input("사용자: ")

    if user_input == "exit":
        break

    print("User: " + user_input)

    response = client.chat.completions.create(
       model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": "너는 사용자를 도와주는 상담사야."},
            {"role": "user", "content": user_input},
        ],
    )

    print("AI: " + response.choices[0].message.content)

User: 내 이름은 조성민이야
AI: 네, 조성민님. 무엇을 도와드릴까요?
User: 내 이름이 뭐야?
AI: 아직 네 이름을 알려주지 않아서 몰라. 뭐라고 부르면 될까?


## 2. 멀티턴 (Multi-turn)
사용자와 AI가 이전의 대화 맥락을 기반으로 여러 번 상호작용을 이어가는 방식입니다.  
AI가 세션 내의 이전 대화(Context)를 메모리에 유지하고 이를 참조하여 답변을 생성합니다.  
특징: "방금 말한 거 다시 설명해줘", "거기서 3번 항목만 조금 더 자세히 작성해줘"와 같은 연속적인 지시가 가능합니다. 사람과 대화하듯 문맥이 이어지는 자연스러운 소통이 가능하지만, 너무 길어지면 모델의 컨텍스트 제한(Context Window)으로 인해 오래된 기억부터 잊어버릴 수 있습니다.

##### chap03/sec02/multi_turn.py

In [4]:
def get_ai_response(messages):
    response = client.chat.completions.create(
        model="gpt-5.6-luna",   # 응답 생성에 사용할 모델 지정
        messages=messages,      # 대화 기록을 입력으로 전달
    )
    return response.choices[0].message.content  # 생성된 응답의 내용 반환

# 초기 시스템 메시지
messages = [
    {
        "role": "system", 
        "content": "너는 사용자를 도와주는 상담사야."
    },  
]

while True:
    user_input = input("사용자: ")

    if user_input == "exit":
        break

    print("User: " + user_input)
    
    # 사용자 메시지를 대화 기록에 추가 
    messages.append({"role": "user", "content": user_input})

    # 대화 기록을 기반으로 AI 응답 가져오기
    ai_response = get_ai_response(messages)  

     # AI 응답 대화 기록에 추가
    messages.append({"role": "assistant", "content": ai_response}) 

    # AI 응답 출력
    print("AI: " + ai_response)  

User: 내 이름은 조성민이야
AI: 알겠습니다, 조성민님. 무엇을 도와드릴까요?
User: 내 이름이 뭐야?
AI: 성민님의 이름은 **조성민**이에요.
